# EcoTrust AI v2.0 - ESG Disclosure Confidence and Integrity Evaluation Pipeline

This Jupyter Notebook implements the 5-layer pipeline of **EcoTrust AI v2.0**.
It is designed to evaluate "Disclosure Confidence" and "Text Integrity" of corporate ESG reports, ensuring strict source auditability through Page-Aware RAG.

**Core System Directive**: This system is strictly an ESG Disclosure Confidence and Integrity Evaluation Platform. It **DOES NOT** detect greenwashing. Instead, it evaluates "Disclosure Confidence" and "Text Integrity", ensuring strict source auditability through Page-Aware RAG. Reflect this positioning across all generated texts.

### Pipeline Layers:
1. **Layer 1: Document Ingestion & Page-Aware Extraction** (using `pdfplumber`)
2. **Layer 2: ESG Validation Gate & Text Preprocessing** (using `jieba` tokenization)
3. **Layer 3: Dynamic Sampling & Smart Chunking** (overcoming the 512-token transformer limit)
4. **Layer 4: FinBERT Semantic Inference & Score Calibration** (calculating Intent Score, Numeracy Density, KPI Richness, and Confidence Score via Sigmoid Contrast Enhancement)
5. **Layer 5: Gen-2 Commitment Extraction & External News Verification** (using Google News RSS and custom lexicon/FinBERT sentiment check)

Plus: **Page-Aware Query Retrieval Simulator** for the RAG Chatbot.

In [ ]:
# Import required packages
import os
import re
import math
import numpy as np
import pdfplumber
import torch
import pandas as pd
import jieba
import urllib.parse
import feedparser
import time
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("PyTorch version:", torch.__version__)
print("Jieba version:", jieba.__version__)

## System Constants, Weights, and Keyword Matrix
Here we define the weight configurations, normalization thresholds, and key matrices used by the evaluation platform.

In [ ]:
# System Weights and Normalization Thresholds
WEIGHT_CONFIG = {
    "W_INTENT": 0.6,          # Weight for FinBERT sentiment intent (I)
    "W_CREDIBILITY": 0.4,     # Weight for document credibility index (C)
    "SUB_W_NUMERACY": 0.45,   # Sub-weight for numeracy density (N_norm)
    "SUB_W_KPI": 0.35,        # Sub-weight for KPI richness (K_norm)
    "SUB_W_RISK": 0.20        # Weight for external news risk score (Re)
}

NORM_THRESHOLDS = {
    "NUMERACY_MAX": 0.20,     # 20% number density for max score
    "KPI_COUNT_MAX": 15,      # 15 distinct ESG keywords for max score
    "RISK_MAX_SCALE": 100.0   # Maximum scale for external risk
}

# ESG Keyword Matrix
ESG_KEYWORDS_MAP = {
    "E": [
        "carbon", "emission", "sustainability", "減碳", "溫室氣體", "再生能源", "水資源", "廢棄物", 
        "net zero", "範疇一", "範疇二", "範疇三", "Scope 1/2/3", "tCO2e", "溫室氣體盤查", "碳中和", 
        "SBTi", "RE100", "能源密集度", "內部碳定價", "廢棄物轉化率", "循環經濟", "再生能源", 
        "綠電採購", "CPPA", "生質能", "生物多樣性", "減塑", "零填埋", "UL2799", "TCFD", 
        "氣候相關財務揭露", "實體風險", "轉型風險", "碳邊境稅", "CBAM", "氣候情境分析", "Scenario Analysis"
    ],
    "S": [
        "human rights", "勞工權益", "職業安全", "社區參與", "人才發展", "供應鏈", "diversity", 
        "離職率", "員工滿意度", "薪資性別平等", "Gender Pay Gap", "多元平等包容", "DEI", 
        "人權盡職調查", "強迫勞動", "結社自由", "ISO 45001", "培訓時數", "育嬰留停復職率", 
        "接班人計畫", "員工持股", "關鍵人才留才", "供應鏈審核", "衝突礦產", "在地採購比率", 
        "社會影響力評估", "隱私保護", "GDPR"
    ],
    "G": [
        "governance", "ethics", "compliance", "董事會", "誠信經營", "風險管理", "反貪腐", "audit", 
        "獨立董事", "董事多元化", "審計委員會", "薪酬委員會", "董事出席率", "董事長與總經理兼任", 
        "反洗錢", "反壟斷", "檢舉機制", "Whistleblowing", "誠信經營手冊", "稅務透明", "政治捐獻", 
        "資安 ISO 27001", "高階主管薪酬連動", "ESG指導委員會", "永續發展長", "CSO"
    ],
    "HARD_METRICS": [
        "KPI", "目標達成率", "基準年", "下降%", "增長%", "驗證", "第三方認證", "ISO", "SASB", 
        "TCFD", "第三方確信", "獨立保證報告", "SGS", "BSI", "DNV", "Deloitte", "PwC", "EY", 
        "KPMG", "會計師核閱", "GRI 2021", "AA1000", "ISAE 3000", "ISO 14064", "ISO 14067", 
        "有限確信", "Limited Assurance", "合理確信", "Reasonable Assurance"
    ]
}

ALL_KEYWORDS = [k for sub in ESG_KEYWORDS_MAP.values() for k in sub]

# ESG Gating Configuration
ESG_MIN_KEYWORD_HITS = 5
ESG_SENTINEL_KEYWORDS = [
    "sustainability", "ESG", "永續", "溫室氣體", "碳排", "碳中和",
    "governance", "董事會", "人才發展", "再生能源", "net zero",
    "TCFD", "GRI", "SASB", "tCO2e", "環境", "誠信", "社會責任",
    "Scope", "範疇", "ISO 14064", "勞工"
]

## Initialize FinBERT Model
We load the FinBERT model (`yiyanghkust/finbert-tone-chinese`). If a local folder exists, we load it locally, otherwise we fall back to the Hugging Face Hub.

In [ ]:
LOCAL_MODEL_DIR = Path("finbert_model")
FALLBACK_MODEL_ID = "yiyanghkust/finbert-tone-chinese"

print("Initializing FinBERT model...")
try:
    if LOCAL_MODEL_DIR.exists() and (LOCAL_MODEL_DIR / "config.json").exists():
        tokenizer = AutoTokenizer.from_pretrained(str(LOCAL_MODEL_DIR), local_files_only=True)
        model = AutoModelForSequenceClassification.from_pretrained(str(LOCAL_MODEL_DIR), local_files_only=True)
        print("Loaded FinBERT model from local directory.")
    else:
        tokenizer = AutoTokenizer.from_pretrained(FALLBACK_MODEL_ID)
        model = AutoModelForSequenceClassification.from_pretrained(FALLBACK_MODEL_ID)
        print("Loaded FinBERT model from Hugging Face Hub.")
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    print("Model successfully loaded on device:", device)
except Exception as e:
    print(f"Error initializing model: {e}")

## Layer 1: Document Ingestion & Page-Aware Extraction
We extract raw text from corporate ESG PDFs page-by-page. Maintaining exact page indexes is crucial for subsequent Page-Aware RAG citation.

In [ ]:
def extract_pdf_pages(pdf_path):
    """
    Extracts text page-by-page from the PDF.
    Keeps track of page numbers to ensure page-aware auditability.
    """
    pages_data = []
    full_text = ""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            total_pages = len(pdf.pages)
            print(f"Total Pages in PDF: {total_pages}")
            for i, page in enumerate(pdf.pages):
                page_text = page.extract_text() or ""
                pages_data.append({
                    "page": i + 1,
                    "text": page_text
                })
                full_text += page_text + "\n"
        print(f"Extraction completed successfully.")
    except Exception as e:
        print(f"Error reading PDF: {e}")
    return pages_data, full_text

# Example Usage (commented out until target file is defined):
# pages_data, full_text = extract_pdf_pages("uploads/your_esg_report.pdf")

## Layer 2: ESG Validation Gate & Text Preprocessing
The pre-check gate validates whether the uploaded document is indeed an ESG report by matching minimum keywords, preventing unrelated corporate reports from clogging the pipeline.

In [ ]:
def is_esg_report(text: str) -> tuple[bool, int, int]:
    """
    Validation gate checks if the document has enough ESG signal.
    Requires at least 5 distinct ESG keyword hits and 2 distinct sentinel hits.
    """
    text_lower = text.lower()
    hits = sum(1 for k in ALL_KEYWORDS if k.lower() in text_lower)
    sentinel_hits = sum(1 for k in ESG_SENTINEL_KEYWORDS if k.lower() in text_lower)
    is_valid = (hits >= ESG_MIN_KEYWORD_HITS and sentinel_hits >= 2)
    return is_valid, hits, sentinel_hits

# Example execution:
# is_valid, hits, sentinel_hits = is_esg_report(full_text)
# if not is_valid: 
#     raise ValueError("Document rejected: Insufficient ESG signals (Not an ESG Report).")

## Layer 3: Dynamic Sampling & Smart Chunking
To overcome FinBERT's 512-token limit, the system avoids simple head truncation or random chunking. Instead, it prioritizes up to 30 sentences containing hard-metric audit terms, then performs uniform sampling across the entire report to select a representative set of 100 sentences.

In [ ]:
def get_smart_chunks(text, max_chunks=100):
    """
    Bypasses FinBERT's 512-token limit by selecting representative sentences.
    1. Splits text by periods or newlines.
    2. Prioritizes up to 30 sentences containing 'HARD_METRICS' (audit keywords).
    3. Uniformly samples remaining sentences to cover the entire report.
    4. Merges, de-duplicates, and caps at max_chunks (100).
    """
    sentences = re.split(r'[。\n]', text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 8]
    
    if not sentences:
        return []

    # Priority extraction of hard-metric sentences
    critical_chunks = [s for s in sentences if any(k in s for k in ESG_KEYWORDS_MAP["HARD_METRICS"])]
    
    # Uniform sampling to cover beginning, middle, and end
    indices = np.linspace(0, len(sentences) - 1, max_chunks).astype(int)
    sampled_chunks = [sentences[i] for i in indices]
    
    # Combine, preserve order and remove duplicates (prioritize critical chunks)
    final_chunks = list(dict.fromkeys(critical_chunks[:30] + sampled_chunks))
    return final_chunks[:max_chunks]

## Layer 4: FinBERT Semantic Inference & Score Calibration
In this layer, we perform batch sentiment inference on the smart chunks and extract numeracy density and KPI diversity. These metrics are mathematically aggregated into a raw integrity score and calibrated via Sigmoid Contrast Enhancement.

In [ ]:
def analyze_intent_sentiment(chunks):
    """
    Evaluates semantic intent across smart chunks.
    Formula: S_chunk = (P_pos - P_neg + 1) / 2
    Intent Score I = Average(S_chunk)
    """
    if not chunks:
        return 0.5
    scores = []
    for chunk in chunks:
        inputs = tokenizer(chunk, return_tensors="pt", truncation=True, max_length=512).to(device)
        with torch.no_grad():
            logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=1).tolist()[0]
        
        if len(probs) == 3:  # [Neutral, Positive, Negative]
            sentiment_score = (probs[1] - probs[2] + 1) / 2
            scores.append(sentiment_score)
        else:
            scores.append(max(probs))
    return float(np.mean(scores))

def calculate_integrity_metrics(text):
    """
    Extracts text integrity metrics:
    1. Numeracy density (Ns) = digit tokens / total tokens
    2. KPI Richness (Kc) = count of distinct ESG keyword matches
    """
    clean_text = re.sub(r'\s+', '', text)
    words = list(jieba.cut(clean_text))
    digits = re.findall(r'\d+(?:\.\d+)?%?', text)
    numeracy_score = len(digits) / max(1, len(words))
    
    found_kpis = sum(1 for k in ALL_KEYWORDS if k in text)
    return numeracy_score, found_kpis

def calculate_integrity_confidence(intent_avg, raw_num, raw_kpi, ext_risk):
    """
    Calculates the final Disclosure Confidence Score Y.
    Formulas:
    N_norm = min(1.0, (raw_num / 0.20) * 1.5)
    K_norm = min(1.0, sqrt(raw_kpi / 15))
    C = (W_num * N_norm) + (W_kpi * K_norm)
    S_raw = (I * W_intent) + (C * W_credibility) + ((1 - Risk/100) * W_risk)
    Y = Sigmoid(S_raw) = 1 / (1 + exp(-10 * (S_raw - 0.5)))
    """
    # 1. Numeracy Normalization
    norm_num = min(1.0, (raw_num / NORM_THRESHOLDS["NUMERACY_MAX"]) * 1.5)
    
    # 2. KPI Count Normalization
    norm_kpi = min(1.0, (raw_kpi / NORM_THRESHOLDS["KPI_COUNT_MAX"]) ** 0.5)
    
    # 3. Credibility Index C
    credibility_index = (WEIGHT_CONFIG["SUB_W_NUMERACY"] * norm_num) + \
                        (WEIGHT_CONFIG["SUB_W_KPI"] * norm_kpi)
    
    # 4. Raw Score calculation
    raw_final = (intent_avg * WEIGHT_CONFIG["W_INTENT"]) + \
                (credibility_index * WEIGHT_CONFIG["W_CREDIBILITY"]) + \
                ((1 - ext_risk / 100.0) * WEIGHT_CONFIG["SUB_W_RISK"])
    
    # 5. Sigmoid Contrast Enhancement (slope k = 10, center = 0.5)
    final_score = 1 / (1 + math.exp(-10 * (raw_final - 0.5)))
    
    return final_score, credibility_index

## Layer 5: Gen-2 Commitment Extraction & External News Verification
Commitments are extracted using grammatical indicators and verified for specific timeframe and quantity markers. High-confidence commitments are validated using Google News RSS feed queries, classified via an ESG dictionary + FinBERT sentiment model.

In [ ]:
PROMISE_VERBS = [
    "承諾", "目標", "計畫", "預計", "將", "擬", "設定", "達成", "實現",
    "commit", "target", "aim", "plan", "pledge", "goal", "objective",
    "intend", "will achieve", "by 20"
]
QUANT_PATTERNS = [
    r'\d+(?:\.\d+)?\s*%',
    r'\d+(?:\.\d+)?\s*tCO2e',
    r'\d+(?:\.\d+)?\s*(噸|萬噸|千噸|公噸)',
    r'\d+(?:\.\d+)?\s*(kWh|MWh|GWh|MW)',
    r'\d+(?:\.\d+)?\s*(億|萬|千)\s*(元|度)',
]
TIMEFRAME_PATTERNS = [
    r'20[2-5]\d\s*年',
    r'by\s*20[2-5]\d',
    r'\d{4}\s*(年|年底|年前|年末)',
    r'(第|第[一二三四])[一二三四五]\s*年',
]

# Custom Lexicon for News Sentiment
POSITIVE_KEYWORDS = [
    "減碳", "碳中和", "零碳", "淨零", "再生能源", "綠電", "節能", "永續",
    "ESG", "CSR", "社會責任", "獲獎", "認證", "ISO", "TCFD", "GRI",
    "榮獲", "優良", "提升", "達標", "通過", "創新", "落實", "承諾",
    "第一", "領先", "優秀", "卓越", "良好", "正面", "積極", "改善",
    "綠色", "環保", "清潔", "低碳", "循環", "公益", "捐助", "志工",
    "多元", "包容", "平等", "人才", "培訓", "效率", "完成", "實現",
]
NEGATIVE_KEYWORDS = [
    "醜聞", "違規", "罰款", "懲處", "訴訟", "告發", "調查", "檢察",
    "汙染", "污染", "排放超標", "廢水", "廢氣", "毒", "違法",
    "裁員", "解雇", "勞資", "糾紛", "抗議", "罷工", "剝削",
    "弊案", "貪汙", "造假", "不實", "虛報", "欺騙", "操縱",
    "下跌", "虧損", "衰退", "縮減", "停產", "停業",
    "事故", "意外", "爆炸", "火災", "洩漏", "breach", "leak",
    "撤資", "退出", "降評", "負面", "批評", "質疑", "危機",
    "困境", "倒閉", "破產", "債務", "違約", "未完成", "延遲", "跳票",
]
STRONG_POSITIVE = {"碳中和", "淨零", "零碳", "榮獲", "ESG", "永續報告", "再生能源", "GRI", "TCFD", "達標", "完成"}
STRONG_NEGATIVE = {"汙染", "污染", "違法", "罰款", "造假", "貪汙", "訴訟", "醜聞", "未完成", "跳票"}

def extract_gen2_commitments(text: str) -> dict:
    sentences = re.split(r'[。\n]', text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 10]

    promises = []
    for s in sentences:
        s_lower = s.lower()
        if any(v in s_lower for v in PROMISE_VERBS):
            has_quant = any(re.search(p, s) for p in QUANT_PATTERNS)
            has_time  = any(re.search(p, s) for p in TIMEFRAME_PATTERNS)

            # Topic classification
            topic_scores = {cat: 0 for cat in ['E', 'S', 'G', 'HARD_METRICS']}
            for cat, kws in ESG_KEYWORDS_MAP.items():
                if any(k.lower() in s_lower for k in kws):
                    topic_scores[cat] += 1
            topic = max(topic_scores, key=topic_scores.get)
            if topic_scores[topic] == 0:
                topic = 'OTHER'

            # Confidence Level
            confidence_level = 'HIGH' if (has_quant and has_time) else \
                               'MED'  if (has_quant or has_time) else 'LOW'

            promises.append({
                'text':       s[:200],
                'has_quant':  has_quant,
                'has_time':   has_time,
                'topic':      topic,
                'confidence': confidence_level,
            })

    total = len(promises)
    quant_rate    = round(sum(1 for p in promises if p['has_quant']) / max(1, total), 4)
    timeframe_rate = round(sum(1 for p in promises if p['has_time'])  / max(1, total), 4)

    topic_dist = {'E': 0, 'S': 0, 'G': 0, 'HARD_METRICS': 0, 'OTHER': 0}
    for p in promises:
        topic_dist[p['topic']] = topic_dist.get(p['topic'], 0) + 1

    high_conf = [p for p in promises if p['confidence'] == 'HIGH'][:8]

    return {
        'total_promises':              total,
        'quant_rate':                  quant_rate,
        'timeframe_rate':              timeframe_rate,
        'topic_distribution':          topic_dist,
        'high_confidence_commitments': high_conf,
    }

def keyword_sentiment(text: str) -> tuple[str, float]:
    pos_score = 0.0
    neg_score = 0.0
    for kw in POSITIVE_KEYWORDS:
        if kw in text:
            pos_score += 2.0 if kw in STRONG_POSITIVE else 1.0
    for kw in NEGATIVE_KEYWORDS:
        if kw in text:
            neg_score += 2.0 if kw in STRONG_NEGATIVE else 1.0
    total = pos_score + neg_score
    if total == 0:
        return "Neutral", 0.60
    ratio = pos_score / total
    if ratio >= 0.6:
        return "Positive", min(round(0.65 + min(pos_score, 5) * 0.06, 4), 0.95)
    elif ratio <= 0.4:
        return "Negative", min(round(0.65 + min(neg_score, 5) * 0.06, 4), 0.95)
    else:
        return "Neutral", round(0.55 + ratio * 0.1, 4)

def fetch_rss_news_verification(company_name: str, high_conf_promises: list, report_year: int):
    verification_results = []
    for action in high_conf_promises:
        text = action['text']
        keywords = []
        esg_terms = ["碳中和", "淨零", "再生能源", "綠電", "TCFD", "SBTi", "RE100", "碳排", "溫室氣體"]
        for term in esg_terms:
            if term in text:
                keywords.append(term)
        num_matches = re.findall(r'\d+(?:\.\d+)?%|\d{4}年', text)
        keywords.extend(num_matches[:1])
        if not keywords:
            keywords.append("永續")
        
        kw_str = ' '.join(keywords[:3])
        query = f"{company_name} {kw_str} {report_year}年"
        print(f"Fetching Google News RSS for query: {query}")
        
        q = urllib.parse.quote(query)
        url = f"https://news.google.com/rss/search?q={q}&hl=zh-TW&gl=TW&ceid=TW:zh-Hant"
        feed = feedparser.parse(url)
        
        news_items = []
        for entry in feed.entries[:3]:
            title = entry.title
            sentiment, score = keyword_sentiment(title)
            news_items.append({
                "title": title,
                "sentiment": sentiment,
                "confidence": score,
                "link": entry.link
            })
        
        verification_results.append({
            "commitment": text,
            "query": query,
            "external_evidence": news_items
        })
        time.sleep(0.3)
    return verification_results

## Page-Aware Query Retrieval Simulator
This mimics the backend RAG mechanism used in `chat_api.php`. It segments the user's query, removes stop words, generates bigrams, calculates term frequency scores per page, and formats the output context with exact page citations.

In [ ]:
def get_chinese_keywords(query: str) -> list[str]:
    query = query.lower()
    # Split by delimiters
    segments = re.split(r'[\s，。、？！,\.\?!：；「」『』（）\(\)\[\]\{\}]+', query)
    segments = [s for s in segments if s]
    
    stop_words = [
        '可以', '給我', '看一看', '有關', '關於', '請', '如何', '什麼', '是多少', '是不是', 
        '有沒有', '以及', '的', '了', '在', '是', '有', '和', '與', '或', '及', '為', 
        '之', '於', '以', '對', '嗎', '呢', '吧', '啊', '這', '那', '哪', '我們', '你們', '他們'
    ]
    domain_terms = [
        '碳排放', '溫室氣體', '減碳', '淨零', '排放量', '範疇', 'roe', 'eps', 
        '永續報告', 'esg', '水資源', '廢棄物', '碳盤查', '確證', '碳中和', '綠電', '再生能源'
    ]
    
    keywords = []
    for clean_segment in segments:
        for sw in stop_words:
            clean_segment = clean_segment.replace(sw, '')
        if len(clean_segment) < 2:
            continue
        keywords.append(clean_segment)
        for term in domain_terms:
            if term in clean_segment:
                keywords.append(term)
        
        # Generate bigrams
        for i in range(len(clean_segment) - 1):
            bigram = clean_segment[i:i+2]
            keywords.append(bigram)
            
    keywords = list(set(keywords))
    keywords = [k for k in keywords if len(k) >= 2]
    return keywords

def select_relevant_pages(pages_list, query, max_pages=40):
    """
    Calculates term frequency score per page and returns sorted context.
    Score(Page_j) = Sum_w(Count(w, Page_j))
    """
    keywords = get_chinese_keywords(query)
    if not keywords:
        keywords = [query.lower()]
        
    scored = []
    for p in pages_list:
        page_text = p['text'].lower()
        if len(page_text) < 20:
            continue
        score = sum(page_text.count(kw) for kw in keywords)
        scored.append({'page': p['page'], 'text': p['text'], 'score': score})
        
    # Sort descending by score
    scored.sort(key=lambda x: x['score'], reverse=True)
    selected = scored[:max_pages]
    
    # Re-sort by page number for reading flow
    selected.sort(key=lambda x: x['page'])
    
    context_str = ""
    for s in selected:
        context_str += f"【第{s['page']}頁】\n{s['text']}\n\n"
        
    if len(context_str) > 30000:
        context_str = context_str[:30000] + "\n...(truncated)"
    return context_str, [s['page'] for s in selected]

## Pipeline Integration and Demo
We construct a pipeline simulator that runs a sample analysis on mock text, mimicking the processing of a real report.

In [ ]:
def run_full_pipeline(pdf_path, company_name, report_year, ext_risk):
    print("=== LAYER 1: Ingesting & Page Indexing ===")
    if not os.path.exists(pdf_path):
        print("Creating mock text for local run (Sample PDF not found)... ")
        mock_text = """
        台泥公司2023年永續報告書。我們承諾將在2030年之前達到減少範疇一與範疇二碳排放50%的目標。
        本公司2023年溫室氣體盤查經過第三方認證，符合ISO 14064標準。範疇一排放量為50000 tCO2e。
        董事會成員多元化比例達30%，符合公司治理守則。董事出席率達95%。
        我們將持續推動循環經濟與再生能源，預計2025年綠電使用率提升至20%。
        """
        pages_data = [{"page": 1, "text": mock_text}, {"page": 2, "text": "獨立董事出席率與薪酬委員會審核細節。"}]
        full_text = mock_text + "\n" + "獨立董事出席率與薪酬委員會審核細節。"
    else:
        pages_data, full_text = extract_pdf_pages(pdf_path)
        
    print("\n=== LAYER 2: ESG Validation Gating ===")
    is_valid, hits, sentinel_hits = is_esg_report(full_text)
    print(f"ESG Gating results: Valid? {is_valid} (Total Hits: {hits}, Sentinel Hits: {sentinel_hits})")
    if not is_valid:
        print("NOT_ESG_REPORT: Aborted.")
        return
        
    print("\n=== LAYER 3: Smart Chunking ===")
    chunks = get_smart_chunks(full_text, max_chunks=100)
    print(f"Selected {len(chunks)} representative chunks.")
    
    print("\n=== LAYER 4: FinBERT Inference & Scoring ===")
    intent_avg = analyze_intent_sentiment(chunks)
    raw_num, raw_kpi = calculate_integrity_metrics(full_text)
    confidence, credibility = calculate_integrity_confidence(intent_avg, raw_num, raw_kpi, ext_risk)
    print(f"FinBERT Intent Avg: {intent_avg:.4f}")
    print(f"Numeracy Score: {raw_num:.4f}, KPI Count: {raw_kpi}")
    print(f"Credibility Index C: {credibility:.4f}")
    print(f"Disclosure Confidence Score Y: {confidence:.4f}")
    
    print("\n=== LAYER 5: Gen-2 Commitment Extraction ===")
    gen2_results = extract_gen2_commitments(full_text)
    print(f"Total Promises Extracted: {gen2_results['total_promises']}")
    print(f"Quantized rate: {gen2_results['quant_rate']:.4f}, Timeframed rate: {gen2_results['timeframe_rate']:.4f}")
    print(f"Topic Distribution: {gen2_results['topic_distribution']}")
    print(f"High-Confidence Promises: {len(gen2_results['high_confidence_commitments'])}")
    
    if gen2_results['high_confidence_commitments']:
        print("\n--- Verifying commitments against external news (Google News RSS) ---")
        try:
            verifications = fetch_rss_news_verification(company_name, gen2_results['high_confidence_commitments'][:1], report_year)
            for v in verifications:
                print(f"\nCommitment: {v['commitment']}")
                print(f"Generated Query: {v['query']}")
                for item in v['external_evidence']:
                    print(f"  - [{item['sentiment']}] {item['title']} (URL: {item['link']})")
        except Exception as e:
            print(f"RSS news fetch skipped or errored: {e}")

# Run the simulator
run_full_pipeline("uploads/mock_report.pdf", "台泥", 2023, 25.0)